# Deep Learning Models Training(CNN,CRNN)

This Notebook is used to train deep learning models to complete Music Genre Classification.
The training model includes:

-CNN (Convolutional Neural Network)
-CRNN (Convolutional Recurrent Neural Network)

The training data uses Mel Spectrogram obtained by preprocessing in advance, and evaluates the model performance in combination with 5-fold Cross Validation.
Finally, use the entire training set to retrain the best model and save the trained parameters for subsequent testing and model evaluation.

## 1 Import Libaraies
The following libraries are imported fordata processing,deep learning,cross validation and model evaluation.

In [46]:
import os
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset, Subset

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

## 2 Gloabal configuration
- Batch Size：The number of samples used for each update of parameters
    Batch Size = 32

- Learning Rate：Adam Optimizer learning rate
    Learning Rate = 0.0005

- Epoch：The number of training sets that the model has been completed
    Epoch = 50

- Number of Classes：Genres in GTZAN dataset：10 Genres。
    Number of Classes = 10

- Device：Autometically detect GPU
    if GPU is available：CUDA
    otherwise：CPU

In [47]:
BATCH_SIZE = 32
LEARNING_RATE = 0.0005
EPOCHS = 50
NUM_CLASSES = 10

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

Mel_dir = "mel"

print("="*60)
print("Training Configuration")
print("="*60)

print(f"Device: {DEVICE}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Epochs: {EPOCHS}")
print(f"Number of Classes: {NUM_CLASSES}")

Training Configuration
Device: cpu
Batch Size: 32
Learning Rate: 0.0005
Epochs: 50
Number of Classes: 10


## 3 Dataset Class
Implement the custom PyTorch data set (GTZANDdataset) to load the preprocessed Mel spectral diagram generated from the GTZAN audio data set.

The data set performs the following tasks:

- Read the file name and type label from the training CSV file

- Match each sample with the corresponding Mel spectral map stored as a NumPy file

- Use zero mean and unit variance normalization to standardize each spectral diagram

- Convert data into PyTorch tensors for deep learning models

This implementation allows PyTorch's "DataLoader" to load data efficiently during training.

In [48]:
class GTZANDdataset(Dataset):
    def __init__(self, mel_dir, x_df, y_df):
        self.mel_dir = mel_dir

        assert len(x_df) == len(y_df)
        assert (x_df["filename"] == y_df["filename"]).all()

        self.genre_to_label = {
            "blues": 0,
            "classical": 1,
            "country": 2,
            "disco": 3,
            "hiphop": 4,
            "jazz": 5,
            "metal": 6,
            "pop": 7,
            "reggae": 8,
            "rock": 9
        }

        self.samples = []
        for i in range(len(x_df)):
            filename = x_df.iloc[i]["filename"]
            genre = y_df.iloc[i]["genre"]
            file_path = os.path.join(mel_dir, genre, filename + ".npy")

            #check if file exists
            if not os.path.exists(file_path):
                raise FileNotFoundError(file_path)

            label = self.genre_to_label[genre]
            self.samples.append((file_path, label))

    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        file_path, label = self.samples[idx]
        mel = np.load(file_path)

        # Standardize each spectrogram
        mel = (mel - mel.mean()) / (mel.std() + 1e-8)

        mel = torch.tensor(mel, dtype=torch.float32)
        mel = mel.unsqueeze(0)
        label = torch.tensor(label, dtype = torch.long)
        return mel, label


## 4 CNN and CRNN models
Two deep learning architectures are implemented:

### 4.1 CNN model
Two convolutional blocks are used to extract local time-frequency characteristics from the Mel spectral map.

Each convolutional block contains:

- **Convolution** ：learn the local audio mode

- **Batch normalization** ：stabilize training and accelerate integration

- **ReLU activation** ：introduce nonlinearity;

- **Max pooling** ：reduce the spatial dimension while retaining important features;

- **Dropout** ：reduce excessive fit.

After feature extraction, the global average pool is used to greatly reduce the number of trainable parameters before finally fully connected to the classifier.

In [49]:
class CNN(nn.Module):
    def __init__(self,num_classes = 10):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(
            in_channels = 1,
            out_channels = 32,
            kernel_size = 3,
            padding = 1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(
            in_channels = 32, 
            out_channels = 64, 
            kernel_size = 3, 
            padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.3)
        )
        
        self.gap = nn.AdaptiveAvgPool2d((1,1))
        self.flatten = nn.Flatten(start_dim = 1)

        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(
            in_features = 64,
            out_features = num_classes
            )
        )
        
        

    def forward(self, x):

        x = self.block1(x)
        x = self.block2(x)

        x = self.gap(x)
        x = self.flatten(x)

        x = self.classifier(x)

        return x

### 4.2 CRNN model
CRNN shares the same convolutional feature extractor as CNN. Then, the extracted feature map is reshaped into a sequence and processed by LSTM to capture time information before classification.

In [50]:
class CRNN(nn.Module):
    def __init__(self,
                 num_classes = 10,
                 freq_bins_after_pool = 8,
                 lstm_hidden_size = 128,
                 lstm_num_layers = 2,
                 dropout = 0.3
                 ):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(
            in_channels = 1,
            out_channels = 32,
            kernel_size = 3,
            padding = 1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(
            in_channels = 32, 
            out_channels = 64, 
            kernel_size = 3, 
            padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.3)
        )

        self.freq_pool = nn.AdaptiveAvgPool2d((freq_bins_after_pool,None))

        self.flatten = nn.Flatten(start_dim = 2)

        lstm_input_size = 64 * freq_bins_after_pool

        self.lstm = nn.LSTM(
            input_size = lstm_input_size,
            hidden_size = lstm_hidden_size,
            num_layers = lstm_num_layers,
            batch_first = True,
            dropout = dropout if lstm_num_layers > 1 else 0
        )


        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(
            in_features = lstm_hidden_size,
            out_features = num_classes
            )
        )

    def forward(self,x):
        x = self.block1(x)
        x = self.block2(x)

        x = self.freq_pool(x)
            
        x = x.permute(0,3,1,2)

        x = self.flatten(x)

        x,_ = self.lstm(x)

        x = x[:, -1, :]

        x = self.classifier(x)

        return x

The two models share the same training strategy, optimizer and evaluation indicators. Therefore, any difference in performance can be attributed mainly to the difference in architecture between CNN and CRNN, rather than the difference in training procedures.

## 5 Training function
Three functions are implemented to perform model training and evaluation

### `train_dl_model()`
This function performs one training epoch. It computes the forward pass, uses CrossEntropy Loss to evaluate loss, performs backpropagation and uses the Adam optimiser to update the model parameters.

In [51]:
def train_dl_model(model, train_loader, optimizer, criterion):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        _, predicted = torch.max(outputs, dim=1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    train_accuracy = 100 * correct / total

    train_loss = total_loss / len(train_loader)

    return train_accuracy , train_loss

### `validation`
The validation function ecaluates the training model on the validation dataset without updationg model parameters.
- Macro F1 scores are calculated because it provides a balanced assessment of all music genres, even if the class distribution is not completely balanced.

In [52]:
def validation(model, validation_loader):
        
        model.eval()
        correct = 0
        total = 0

        all_preds = []
        all_labels = []

        with torch.no_grad():

            for images, labels in validation_loader:
                images = images.to(DEVICE)
                labels = labels.to(DEVICE)

                outputs = model(images)

                _, predicted = torch.max(outputs, dim=1)

                total += labels.size(0)

                correct += (predicted == labels).sum().item()

                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        validation_accuracy = 100 * correct  / total

        validation_f1 = f1_score(
            all_labels,
            all_preds,
            average="macro"
        )

        return validation_accuracy, validation_f1

### `cross_validation()`
In order to improve the stability and classification performance of model training,our project adopts：5-Fold Cross Validation
Workflow：Training Dataset-->Split into 5 Folds-->4 Folds Training-->1 Fold Validation-->Repeat 5 Times
- The macro F1 score is used as an evaluation indicator because it gives equal attention to each music genre.

In [53]:
def cross_validation(train_dataset, Model):

    labels = [label for _, label in train_dataset.samples]
    kf = StratifiedKFold(
        n_splits = 5, 
        shuffle = True, 
        random_state = 42
    )
    
    fold_f1_scores = []

    fold_results = []

    for fold, (train_idx, validation_idx) in enumerate(kf.split(train_dataset.samples, labels)):

        print(f"Fold {fold + 1}")
        best_f1 = 0

        train_subset = Subset(train_dataset, train_idx)
        validation_subset = Subset(train_dataset,validation_idx)

        train_loader = DataLoader(
            train_subset,
            batch_size=BATCH_SIZE,
            shuffle=True
        )

        validation_loader = DataLoader(
            validation_subset,
            batch_size = BATCH_SIZE,
            shuffle = False
        )

        model = Model(NUM_CLASSES).to(DEVICE)

        criterion = nn.CrossEntropyLoss()

        optimizer = Adam(
            model.parameters(),
            lr=LEARNING_RATE,
            weight_decay=1e-4
        )

        for epoch in range(EPOCHS):
            train_accuracy , train_loss = train_dl_model(model, train_loader, optimizer, criterion)
            validation_accuracy, validation_f1 = validation(model, validation_loader)

            if validation_f1 > best_f1:
                best_f1 = validation_f1

            print(
                f"Epoch [{epoch+1}/{EPOCHS}] "
                f"Loss: {train_loss:.4f} "
                f"Train Accuracy: {train_accuracy:.2f}% "
                f"Validation Accuracy: {validation_accuracy:.2f}% "
                f"Macro F1:{validation_f1:.4f}"
            )

        fold_f1_scores.append(best_f1)

        fold_results.append({
            "Model": Model.__name__,
            "Fold": fold + 1,
            "Macro F1": round(best_f1, 4)
        })
        
    avg_f1 = sum(fold_f1_scores) / len(fold_f1_scores)
    
    print(f"Average Macro F1 :{avg_f1:.4f}") 
   
    return avg_f1, fold_results

## 6 Load Traing Data
Load the pre-generated training feature and label files, then construct the PyTorch dataset for model training.

In [54]:
# Load training CSV files
x_train = pd.read_csv("X_train.csv")
y_train = pd.read_csv("y_train.csv")

# Create dataset
train_dataset = GTZANDdataset(Mel_dir, x_train, y_train)

print("Dataset Information")
print("-" * 40)
print(f"Training samples : {len(train_dataset)}")

labels = [label for _, label in train_dataset.samples]
print(f"Number of classes: {len(set(labels))}")

sample, label = train_dataset[0]
print(f"Input shape      : {tuple(sample.shape)}")
print(f"Sample label     : {label.item()}")

Dataset Information
----------------------------------------
Training samples : 804
Number of classes: 10
Input shape      : (1, 128, 1292)
Sample label     : 0


## 7 CNN training

 ### 7.1 CNN training process
The CNN model is trained using five-fold stratified cross validation.  
- For each fold, the model is trained for 50 epochs using the Adam optimizer and CrossEntropyLoss.  
- The best Macro F1 score on the validation set is recorded for each fold, and the average Macro F1 across all folds is used to evaluate the model.

In [ ]:
print("=" * 60)
print("CNN Training")
print("=" * 60)

print(f"Model : CNN")
print(f"Device: {DEVICE}")
print(f"Epochs: {EPOCHS}")
print(f"Batch Size: {BATCH_SIZE}")

cnn_avg_f1, cnn_fold_results = cross_validation(
    train_dataset,
    CNN
)

CNN Training
Model : CNN
Device: cpu
Epochs: 50
Batch Size: 32
Fold 1


### 7.2 Training results
The following table summarises the best Macro F1 score achieved in each fold during five-fold cross validation. The average Macro F1 is reported as the overall training performance of the CNN model.

In [ ]:
cnn_results_df = pd.DataFrame(cnn_fold_results)

print("CNN Cross Validation Results")
display(cnn_results_df)

print(f"\nAverage CNN Macro F1: {cnn_avg_f1:.4f}")

CNN Cross Validation Results


,Model,Fold,Macro F1
0,CNN,1,0.4533
1,CNN,2,0.4602
2,CNN,3,0.4861
3,CNN,4,0.4650
4,CNN,5,0.4787



Average CNN Macro F1: 0.4687


## 8 CRNN training

### 8.1 CRNN training process
The CRNN model follows the same five-fold stratified cross validation procedure as the CNN model. The convolutional layers first extract spatial features from Mel spectrograms, and the LSTM captures temporal dependencies before classification.

In [ ]:
print("=" * 60)
print("CRNN Training")
print("=" * 60)

print(f"Model : CRNN")
print(f"Device: {DEVICE}")
print(f"Epochs: {EPOCHS}")
print(f"Batch Size: {BATCH_SIZE}")

crnn_avg_f1, crnn_fold_results = cross_validation(
    train_dataset,
    CRNN
)

CRNN Training
Model : CRNN
Device: cpu
Epochs: 50
Batch Size: 32
Fold 1
Epoch [1/50] Loss: 2.2559 Train Accuracy: 19.75% Validation Accuracy: 26.71% Macro F1:0.1220
Epoch [2/50] Loss: 2.0695 Train Accuracy: 29.39% Validation Accuracy: 28.57% Macro F1:0.1670
Epoch [3/50] Loss: 1.8513 Train Accuracy: 34.68% Validation Accuracy: 37.27% Macro F1:0.2982
Epoch [4/50] Loss: 1.7445 Train Accuracy: 37.64% Validation Accuracy: 35.40% Macro F1:0.2834
Epoch [5/50] Loss: 1.7631 Train Accuracy: 38.88% Validation Accuracy: 39.75% Macro F1:0.3573
Epoch [6/50] Loss: 1.6613 Train Accuracy: 41.52% Validation Accuracy: 41.61% Macro F1:0.3834
Epoch [7/50] Loss: 1.5434 Train Accuracy: 44.17% Validation Accuracy: 32.92% Macro F1:0.2716
Epoch [8/50] Loss: 1.6581 Train Accuracy: 36.86% Validation Accuracy: 44.72% Macro F1:0.3904
Epoch [9/50] Loss: 1.6634 Train Accuracy: 38.72% Validation Accuracy: 41.61% Macro F1:0.3413
Epoch [10/50] Loss: 1.5535 Train Accuracy: 44.32% Validation Accuracy: 47.83% Macro F1:0.41

### 8.2 CRNN training results
The best Macro F1 score from each fold is summarised below. The average Macro F1 is used as the overall performance indicator of the CRNN model.

In [ ]:
crnn_results_df = pd.DataFrame(crnn_fold_results)

print("CRNN Cross Validation Results")
display(crnn_results_df)

print(f"\nAverage CRNN Macro F1: {crnn_avg_f1:.4f}")

CRNN Cross Validation Results


,Model,Fold,Macro F1
0,CRNN,1,0.5778
1,CRNN,2,0.5468
2,CRNN,3,0.5515
3,CRNN,4,0.5497
4,CRNN,5,0.5390



Average CRNN Macro F1: 0.5530
